# Rent Estimation Model Training

Reference notebook for the HomeLink AI Engine (`ai/` microservice).

> The canonical training pipeline now lives in `ai/src/`: `generate_dataset.py` builds `ai/data/ethiopia_housing_data.csv` and `train_rent_model.py` fits a `RandomForestRegressor` and saves `ai/models/rent_model.joblib`. This notebook is kept as an interactive exploration/training workspace for the same feature contract.

## Goal
Train a scikit-learn pipeline that predicts monthly rent in ETB from:

- `area_sqm` (numeric)
- `bedrooms` (numeric)
- `bathrooms` (numeric)
- `has_water_tank` (numeric, 0/1)
- `has_generator` (numeric, 0/1)
- `is_furnished` (numeric, 0/1)
- `subcity` (categorical, one-hot encoded)

## Training contract
The pipeline is saved to `../models/rent_model.joblib` and is loaded by `ai/src/rent_estimator.py`. Until a model is present, the service serves a deterministic heuristic baseline so it remains fully operational.

> **Feature order matters.** The serving module builds a single-row DataFrame with columns `[subcity, bedrooms, bathrooms, area_sqm, has_water_tank, has_generator, is_furnished]` (subcity lowercased). The pipeline must be trained on the same column layout.

In [ ]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

AI_DIR = Path.cwd().parent
MODELS_DIR = AI_DIR / "models"
DATA_PATH = AI_DIR / "data" / "ethiopia_housing_data.csv"

NUMERIC_COLUMNS = ["bedrooms", "bathrooms", "area_sqm",
                   "has_water_tank", "has_generator", "is_furnished"]
CATEGORICAL_COLUMNS = ["subcity"]
FEATURE_COLUMNS = CATEGORICAL_COLUMNS + NUMERIC_COLUMNS

In [ ]:
# Load the synthetic dataset generated by ai/src/generate_dataset.py.
data = pd.read_csv(DATA_PATH)
data["subcity"] = data["subcity"].astype(str).str.strip().str.lower()
bool_cols = ["has_water_tank", "has_generator", "is_furnished"]
for col in bool_cols:
    data[col] = data[col].astype(str).str.strip().str.lower().map(
        lambda v: 1 if v in {"1", "true", "yes"} else 0
    )

print(f"Loaded {len(data):,} rows from {DATA_PATH}")
print("Columns:", list(data.columns))
data.head()

X = data[FEATURE_COLUMNS]
y = data["rent_price_etb"]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_COLUMNS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLUMNS),
    ]
)
model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("regressor", GradientBoostingRegressor(n_estimators=200, random_state=42)),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model.fit(X_train, y_train)

print("R2 (test):", round(model.score(X_test, y_test), 4))

In [ ]:
MODELS_DIR.mkdir(exist_ok=True)
joblib.dump(model, MODELS_DIR / "rent_model.joblib")
print(f"Saved pipeline to {MODELS_DIR / 'rent_model.joblib'}")